# Notebook 7: Performance Measurement

"If you can't measure it, you can't improve it." In this notebook, you will learn:

- CUDA events for precise GPU timing
- Measuring and interpreting memory bandwidth
- Understanding the roofline model at a high level
- Arithmetic intensity and compute vs memory bound
- Profiling with `nsys` and `ncu`
- Warmup and benchmarking methodology

In [ ]:
%load_ext nvcc4jupyter

## 7.1 CUDA Events — The Right Way to Time GPU Code

**Never use CPU timers** (`clock()`, `std::chrono`) for GPU timing. The kernel launch is asynchronous — a CPU timer measures launch overhead, not actual execution.

**CUDA events** are timestamps recorded on the GPU's timeline:

```cpp
cudaEvent_t start, stop;
cudaEventCreate(&start);
cudaEventCreate(&stop);

cudaEventRecord(start);         // Record timestamp on GPU
kernel<<<grid, block>>>(...);
cudaEventRecord(stop);          // Record timestamp after kernel
cudaEventSynchronize(stop);     // Wait for stop event

float ms;
cudaEventElapsedTime(&ms, start, stop);  // Elapsed time in milliseconds
```

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void vector_add(const float* a, const float* b, float* c, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (int i = idx; i < N; i += stride)
        c[i] = a[i] + b[i];
}

int main() {
    const int N = 1 << 24;  // 16M
    const size_t bytes = N * sizeof(float);

    float *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));

    // Initialize on device (kernel to fill with values)
    CUDA_CHECK(cudaMemset(d_a, 0, bytes));
    CUDA_CHECK(cudaMemset(d_b, 0, bytes));

    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;

    // =========================================
    // Warmup — first kernel launch has overhead
    // =========================================
    vector_add<<<grid_size, block_size>>>(d_a, d_b, d_c, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    // =========================================
    // Proper timing with CUDA events
    // =========================================
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Run multiple times and average
    const int RUNS = 100;
    cudaEventRecord(start);
    for (int i = 0; i < RUNS; i++) {
        vector_add<<<grid_size, block_size>>>(d_a, d_b, d_c, N);
    }
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float total_ms;
    cudaEventElapsedTime(&total_ms, start, stop);
    float avg_ms = total_ms / RUNS;

    // Bandwidth calculation
    float total_bytes = 3.0f * bytes;  // Read a, read b, write c
    float bandwidth_gb_s = total_bytes / (avg_ms / 1000.0f) / 1e9;

    printf("Vector add: %d elements (%.1f MB)\n", N, bytes / 1e6);
    printf("Average over %d runs: %.3f ms\n", RUNS, avg_ms);
    printf("Effective bandwidth: %.1f GB/s\n", bandwidth_gb_s);

    // What's the theoretical peak?
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    float peak_bw = 2.0f * prop.memoryClockRate * 1e3 * (prop.memoryBusWidth / 8) / 1e9;
    printf("\nTheoretical peak bandwidth: %.1f GB/s\n", peak_bw);
    printf("Achieved: %.1f%% of peak\n", 100.0f * bandwidth_gb_s / peak_bw);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return 0;
}

### Benchmarking Best Practices

1. **Always warmup** — The first kernel launch includes JIT compilation, context setup, etc.
2. **Run multiple iterations** — Average over 10-100 runs for stable results.
3. **Use CUDA events** — Never CPU timers for kernel timing.
4. **Report bandwidth** — More meaningful than raw milliseconds for memory-bound kernels.
5. **Compare to peak** — Know what fraction of theoretical bandwidth you achieve.

## 7.2 Memory Bandwidth — The Key Metric

For most CUDA kernels, the bottleneck is **memory bandwidth**, not compute.

```
Effective bandwidth = bytes_read_and_written / kernel_time
```

**Typical peak bandwidths:**
| GPU | Peak BW |
|-----|--------|
| GTX 1080 | 320 GB/s |
| RTX 3090 | 936 GB/s |
| V100 | 900 GB/s |
| A100 | 2039 GB/s |
| H100 | 3350 GB/s |

A well-optimized memory-bound kernel achieves 70-80% of peak bandwidth.

## 7.3 Arithmetic Intensity — Are You Compute or Memory Bound?

**Arithmetic intensity** = FLOPs / bytes transferred

```
Vector add:  c[i] = a[i] + b[i]
  - 1 FLOP per element (the addition)
  - 12 bytes per element (read a, read b, write c = 3 × 4 bytes)
  - Arithmetic intensity = 1/12 = 0.083 FLOP/byte → MEMORY BOUND

Matrix multiply:  C[i][j] = sum_k(A[i][k] * B[k][j])
  - 2N FLOPs per output element (N multiplies + N adds)
  - ~8 bytes per output element (amortized with tiling)
  - Arithmetic intensity = 2N/8 = N/4 FLOP/byte → COMPUTE BOUND (for large N)
```

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

// Low arithmetic intensity: 1 FLOP / 12 bytes
__global__ void vec_add(const float* a, const float* b, float* c, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) c[i] = a[i] + b[i];
}

// High arithmetic intensity: many FLOPs / 8 bytes
__global__ void heavy_compute(const float* in, float* out, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float x = in[i];
        // 50+ FLOPs per element
        for (int j = 0; j < 50; j++) {
            x = sinf(x) * cosf(x) + 0.1f;
        }
        out[i] = x;
    }
}

float benchmark(void (*launcher)(float*, float*, float*, int, int, int),
                float* d_a, float* d_b, float* d_c,
                int N, int grid, int block) {
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warmup
    launcher(d_a, d_b, d_c, N, grid, block);
    cudaDeviceSynchronize();

    int RUNS = 50;
    cudaEventRecord(start);
    for (int i = 0; i < RUNS; i++)
        launcher(d_a, d_b, d_c, N, grid, block);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    return ms / RUNS;
}

void launch_add(float* a, float* b, float* c, int N, int grid, int block) {
    vec_add<<<grid, block>>>(a, b, c, N);
}

void launch_heavy(float* a, float* b, float* c, int N, int grid, int block) {
    heavy_compute<<<grid, block>>>(a, c, N);
}

int main() {
    const int N = 1 << 22;  // 4M elements
    const size_t bytes = N * sizeof(float);

    float *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));
    CUDA_CHECK(cudaMemset(d_a, 0, bytes));
    CUDA_CHECK(cudaMemset(d_b, 0, bytes));

    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;

    float ms_add = benchmark(launch_add, d_a, d_b, d_c, N, grid_size, block_size);
    float ms_heavy = benchmark(launch_heavy, d_a, d_b, d_c, N, grid_size, block_size);

    float bw_add = 3.0f * bytes / (ms_add / 1000.0f) / 1e9;
    float bw_heavy = 2.0f * bytes / (ms_heavy / 1000.0f) / 1e9;

    printf("%-20s | %8s | %10s | %s\n", "Kernel", "Time", "Bandwidth", "Bottleneck");
    printf("---------------------|----------|------------|----------\n");
    printf("%-20s | %6.3f ms | %7.1f GB/s | Memory\n", "Vector add", ms_add, bw_add);
    printf("%-20s | %6.3f ms | %7.1f GB/s | Compute\n", "Heavy compute", ms_heavy, bw_heavy);
    printf("\nNote: Heavy compute is slower but achieves lower bandwidth\n");
    printf("because it's compute-bound — the GPU is doing math, not waiting for memory.\n");

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    return 0;
}

## 7.4 The Roofline Model (Simplified)

The roofline model tells you whether a kernel is limited by memory or compute:

```
Performance (GFLOPS)
  │
  │         ╱─────────── Peak compute (GFLOPS)
  │        ╱
  │       ╱
  │      ╱   ← Kernel sits here based on arithmetic intensity
  │     ╱
  │    ╱
  │   ╱
  │  ╱  Memory bandwidth slope
  │ ╱
  └─────────────────────────────────────
  0     Arithmetic Intensity (FLOP/byte)
        ↑ memory-bound    ↑ compute-bound
```

- **Left of the ridge:** Memory-bound. Optimize memory access patterns.
- **Right of the ridge:** Compute-bound. Optimize arithmetic efficiency.
- **On the ridge:** Balanced. Best utilization.

Vector addition (0.083 FLOP/byte) is firmly in the memory-bound region.
Matrix multiplication (large N) is in the compute-bound region.

## 7.5 Measuring Transfer Overhead

For a complete picture, measure all three phases: transfer in, compute, transfer out.

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call)                                                      \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                      \
                    __FILE__, __LINE__, cudaGetErrorString(err));              \
            exit(EXIT_FAILURE);                                                \
        }                                                                      \
    } while (0)

__global__ void saxpy(float a, const float* x, float* y, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = a * x[i] + y[i];
}

int main() {
    int sizes[] = {1<<16, 1<<18, 1<<20, 1<<22, 1<<24};
    const char* labels[] = {"64K", "256K", "1M", "4M", "16M"};

    printf("%6s | %10s | %10s | %10s | %10s | %s\n",
           "Size", "H->D (ms)", "Kernel", "D->H (ms)", "Total", "Kernel %");
    printf("-------|------------|------------|------------|------------|--------\n");

    for (int s = 0; s < 5; s++) {
        int N = sizes[s];
        size_t bytes = N * sizeof(float);

        float* h_x = (float*)malloc(bytes);
        float* h_y = (float*)malloc(bytes);
        for (int i = 0; i < N; i++) { h_x[i] = 1.0f; h_y[i] = 2.0f; }

        float *d_x, *d_y;
        CUDA_CHECK(cudaMalloc(&d_x, bytes));
        CUDA_CHECK(cudaMalloc(&d_y, bytes));

        cudaEvent_t t0, t1, t2, t3;
        cudaEventCreate(&t0);
        cudaEventCreate(&t1);
        cudaEventCreate(&t2);
        cudaEventCreate(&t3);

        int bs = 256, gs = (N + bs - 1) / bs;

        cudaEventRecord(t0);
        cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice);
        cudaMemcpy(d_y, h_y, bytes, cudaMemcpyHostToDevice);
        cudaEventRecord(t1);
        saxpy<<<gs, bs>>>(2.0f, d_x, d_y, N);
        cudaEventRecord(t2);
        cudaMemcpy(h_y, d_y, bytes, cudaMemcpyDeviceToHost);
        cudaEventRecord(t3);
        cudaEventSynchronize(t3);

        float h2d, kernel, d2h;
        cudaEventElapsedTime(&h2d, t0, t1);
        cudaEventElapsedTime(&kernel, t1, t2);
        cudaEventElapsedTime(&d2h, t2, t3);
        float total = h2d + kernel + d2h;

        printf("%6s | %10.3f | %10.3f | %10.3f | %10.3f | %5.1f%%\n",
               labels[s], h2d, kernel, d2h, total,
               100.0f * kernel / total);

        cudaEventDestroy(t0); cudaEventDestroy(t1);
        cudaEventDestroy(t2); cudaEventDestroy(t3);
        cudaFree(d_x); cudaFree(d_y);
        free(h_x); free(h_y);
    }

    printf("\nFor memory-bound kernels, transfers dominate total time.\n");
    printf("This is why minimizing transfers is critical!\n");
    return 0;
}

## 7.6 Profiling Tools

Beyond manual timing, NVIDIA provides profiling tools:

### Nsight Systems (`nsys`) — Timeline profiler
```bash
nsys profile --stats=true ./my_program
```
Shows: kernel launches, memory transfers, API calls on a timeline. Great for finding bottlenecks at the application level.

### Nsight Compute (`ncu`) — Kernel profiler
```bash
ncu --set full ./my_program
```
Shows: per-kernel metrics — occupancy, memory throughput, instruction throughput, warp stalls. Great for optimizing individual kernels.

### Key metrics from `ncu`:

| Metric | What it tells you |
|--------|------------------|
| **Occupancy** | % of max threads active on each SM |
| **Memory throughput** | % of peak bandwidth used |
| **Compute throughput** | % of peak FLOPS used |
| **Warp stall reasons** | Why threads are waiting (memory, execution, sync) |

## 7.7 Occupancy

**Occupancy** = active warps / maximum warps per SM

Higher occupancy generally means better latency hiding (while one warp waits for memory, another can execute). But it's not the only factor — sometimes lower occupancy with better data reuse (shared memory) wins.

Factors that limit occupancy:
1. **Threads per block** — too few wastes SM capacity
2. **Registers per thread** — complex kernels use more registers, limiting active threads
3. **Shared memory per block** — large shared memory reduces concurrent blocks

CUDA provides an API to query theoretical occupancy:

In [ ]:
%%cuda
#include <cstdio>

__global__ void simple_kernel(float* data, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) data[i] *= 2.0f;
}

int main() {
    int block_sizes[] = {32, 64, 128, 256, 512, 1024};

    printf("%10s | %12s | %10s\n", "Block size", "Max blocks/SM", "Occupancy");
    printf("-----------|--------------|----------\n");

    for (int i = 0; i < 6; i++) {
        int bs = block_sizes[i];
        int max_blocks;

        cudaOccupancyMaxActiveBlocksPerMultiprocessor(
            &max_blocks, simple_kernel, bs, 0);

        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, 0);
        int max_warps_per_sm = prop.maxThreadsPerMultiProcessor / 32;
        int active_warps = max_blocks * (bs / 32);
        float occupancy = (float)active_warps / max_warps_per_sm * 100.0f;

        printf("%10d | %12d | %8.1f%%\n", bs, max_blocks, occupancy);
    }

    // Auto-select optimal block size
    int min_grid, opt_block;
    cudaOccupancyMaxPotentialBlockSize(&min_grid, &opt_block, simple_kernel, 0, 0);
    printf("\nOptimal block size (auto): %d (min grid: %d)\n", opt_block, min_grid);

    return 0;
}

## 7.8 Performance Checklist for Beginners

When your kernel is slow, check these in order:

1. **Are transfers dominating?** → Minimize data movement, use pinned memory
2. **Is memory bandwidth low?** → Check for uncoalesced access, use shared memory
3. **Is occupancy too low?** → Reduce registers/shared memory, adjust block size
4. **Is there warp divergence?** → Reorganize branches to align with warp boundaries
5. **Is arithmetic too slow?** → Use fast math (`__fmul_rn`), `float` instead of `double`

For most beginner kernels, items 1 and 2 are the bottleneck.

## 7.9 Key Takeaways

1. **CUDA events** are the correct way to time GPU code — never use CPU timers
2. **Always warmup** before timing — first launch has overhead
3. **Report bandwidth** (GB/s), not just time — it's meaningful across data sizes
4. **Arithmetic intensity** determines if you're memory or compute bound
5. **Include transfer time** for a realistic picture of GPU advantage
6. **Occupancy** matters but isn't everything — 50%+ is usually fine
7. **Use profiling tools** (`nsys`, `ncu`) for real optimization work
8. **Most beginner kernels** are memory-bound — optimize data access first

## Exercises

**Exercise 7.1:** Write a benchmark that measures the bandwidth of `cudaMemcpy` for sizes from 1 KB to 1 GB. Plot the results (save to file and use Python/matplotlib to visualize). At what size does bandwidth saturate?

**Exercise 7.2:** Compare the performance of vector add using `float` vs `double`. How does the effective bandwidth change? Why?

**Exercise 7.3:** Write a compute-bound kernel that does 100 `sinf()` calls per element. What's the achieved FLOPS? How does it compare to vector add?

In [ ]:
%%cuda
// Exercise 7.1 — Bandwidth vs transfer size
#include <cstdio>
#include <cstdlib>

int main() {
    printf("%12s | %10s | %10s\n", "Size", "Time (ms)", "BW (GB/s)");
    printf("-------------|------------|----------\n");

    for (int power = 10; power <= 28; power += 2) {
        size_t bytes = (size_t)1 << power;
        // TODO: Allocate, time cudaMemcpy H->D, compute bandwidth
    }
    return 0;
}

---

## Congratulations!

You've completed the beginner CUDA C++ course! You now know:

- The CUDA programming model (host/device, kernels, threads)
- Thread hierarchy (grids, blocks, warps)
- Memory management (explicit, unified, pinned)
- Writing real kernels (vector add, matrix ops)
- Error handling and debugging
- Performance measurement and basic optimization

### What's Next? (Intermediate Topics)

- **Shared memory** — Fast on-chip memory for data reuse within a block
- **Memory coalescing** — Optimizing global memory access patterns
- **Reduction patterns** — Parallel sum, max, min across threads
- **Streams and async** — Overlapping compute with data transfers
- **Atomic operations** — Thread-safe read-modify-write
- **Cooperative groups** — Flexible thread synchronization
- **Tiled matrix multiplication** — The classic shared memory optimization